# Fase 1: revisión del dataset SIPaKMeD

Antes de entrenar cualquier cosa hay que saber con qué estamos trabajando. Esta primera parte
no entrena nada, solo revisa el dataset para responder tres preguntas:

1. ¿Qué archivos hay y cómo están organizados?
2. ¿De dónde salió cada foto de célula? (esto resulta ser lo más importante)
3. ¿Las anotaciones de los contornos están completas y bien puestas?

El dataset son fotos de células de papanicolaou divididas en cinco tipos. La idea del proyecto
es clasificarlas, pero antes de eso queremos asegurarnos de que la forma de evaluar el modelo
sea honesta, y eso se decide justamente acá.

## Importar el dataset

Montamos Drive para poder guardar ahí los resultados y no perderlos cuando se cierre la sesión.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Para bajar el dataset hace falta el token de Kaggle. Lo guardamos en los *Secrets* de Colab
(el icono de llave en la barra izquierda) en vez de escribirlo en una celda, así no queda
expuesto si compartimos el notebook.

In [ ]:
!pip install -q -U kagglehub

import os
from google.colab import userdata

# El token se lee desde Secrets, nunca se escribe directo acá
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

Descarga del dataset. Son unos 6.4 GB comprimidos que se descomprimen a 9.2 GB, así que tarda
un par de minutos. Se guarda en el disco de la máquina de Colab, no en Drive.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")
print("Dataset en:", path)

!du -sh "{path}"
!ls "{path}"

## ¿Cómo está organizado?

Recorremos las carpetas contando archivos. Nos interesan dos extensiones: los `.bmp` son las
imágenes y los `.dat` son los contornos dibujados a mano por los especialistas, que marcan
dónde está el borde de la célula y dónde el núcleo.

In [ ]:
import os

for root, dirs, files in os.walk(path):
    nivel = root.replace(path, "").count(os.sep)
    if nivel <= 2:
        n_bmp = len([f for f in files if f.lower().endswith(".bmp")])
        n_dat = len([f for f in files if f.lower().endswith(".dat")])
        print("  " * nivel + os.path.basename(root) + f"/   [bmp: {n_bmp}, dat: {n_dat}]")

# De paso revisamos que no nos quedemos sin disco
!df -h /content | tail -1

Los `.bmp` de este nivel suman 966 y los `.dat` suman 8098. Ese segundo número dividido para 2
da 4049, que es la cantidad de células del dataset. O sea que hay dos contornos por célula
(uno del borde y otro del núcleo), y las 966 imágenes son las láminas completas de las que se
recortaron esas células.

Traducido: las 4049 células no vienen de 4049 pacientes distintos, vienen de menos de mil
láminas. Eso hay que tenerlo muy presente más adelante.

Miramos por dentro una clase cualquiera para entender el patrón de los nombres.

In [ ]:
carpeta = os.path.join(path, "im_Parabasal", "im_Parabasal")

print("Contenido del nivel 2:")
print(sorted(os.listdir(carpeta))[:25])
print("...")

print("\nSubcarpetas dentro:")
for d in sorted(os.listdir(carpeta)):
    ruta = os.path.join(carpeta, d)
    if os.path.isdir(ruta):
        archivos = os.listdir(ruta)
        print(f"  {d}/  -> {len(archivos)} archivos, ejemplos: {sorted(archivos)[:6]}")

Ahora está claro:

- En el nivel de arriba está la lámina entera (`001.bmp`) con los contornos de todas sus células.
- Dentro de `CROPPED/` está cada célula recortada (`001_01.bmp`) con sus dos contornos
  (`001_01_cyt.dat` para el borde y `001_01_nuc.dat` para el núcleo).

**Lo importante:** el número de adelante del nombre dice de qué lámina salió cada célula.
`001_01` y `001_02` son dos células vecinas del mismo cúmulo. Vamos a necesitar eso.

Y así se ve un archivo de contorno por dentro: una coordenada por línea, separadas por coma.

In [ ]:
import glob

dat = sorted(glob.glob(os.path.join(carpeta, "*.dat")))[0]
print("Archivo:", os.path.basename(dat))
!head -5 "{dat}"
!wc -l "{dat}"

## Armar el índice

Recorremos las cinco clases y construimos una tabla con una fila por célula: qué clase es,
de qué lámina viene, dónde está la imagen y dónde están sus dos contornos.

Al armar la columna `lamina` le pegamos el nombre de la clase adelante. Esto es a propósito:
la lámina `001` de Parabasal y la `001` de Metaplásica son láminas distintas que por casualidad
tienen el mismo número, y si no las diferenciamos el programa las trataría como si fueran la misma.

In [ ]:
import glob, os, re, cv2
import pandas as pd

CLASES = ["im_Superficial-Intermediate", "im_Parabasal", "im_Koilocytotic",
          "im_Dyskeratotic", "im_Metaplastic"]

filas = []
for c in CLASES:
    cropped = os.path.join(path, c, c, "CROPPED")
    for img in sorted(glob.glob(os.path.join(cropped, "*.bmp"))):
        stem = os.path.splitext(os.path.basename(img))[0]   # ej. 001_01
        filas.append({
            "clase": c.replace("im_", ""),
            # el numero antes del guion bajo identifica la lamina de origen
            "lamina": c.replace("im_", "") + "_" + stem.split("_")[0],
            "stem": stem,
            "img": img,
            "cyt": os.path.join(cropped, stem + "_cyt.dat"),
            "nuc": os.path.join(cropped, stem + "_nuc.dat"),
        })

df = pd.DataFrame(filas)

# Verificamos que a ninguna celula le falte un contorno
df["existe_cyt"] = df["cyt"].apply(os.path.exists)
df["existe_nuc"] = df["nuc"].apply(os.path.exists)

print("Células totales:", len(df))
print(df["clase"].value_counts(), "\n")
print("Láminas únicas:", df["lamina"].nunique())
print("Células por lámina — media/mín/máx:",
      round(df.groupby("lamina").size().mean(), 1),
      df.groupby("lamina").size().min(),
      df.groupby("lamina").size().max())
print("Contornos faltantes:", (~df.existe_cyt).sum(), (~df.existe_nuc).sum())

Salen 4049 células repartidas en 966 láminas, con la misma cantidad por clase que reporta el
paper original del dataset. Cero contornos faltantes, o sea que la anotación está completa.

El dato que hay que subrayar es el promedio: **4.2 células por lámina**, y hay una lámina con 26.
Cada célula tiene en promedio tres o cuatro hermanas que salieron del mismo cúmulo y que se
parecen muchísimo entre ellas.

Antes de seguir, una comprobación técnica: ¿los contornos están en coordenadas del recorte o de
la lámina completa? Si el contorno se sale del tamaño de la imagen recortada, habría que
corregirlo restando un desplazamiento.

In [ ]:
fila = df.iloc[0]
h, w = cv2.imread(fila["img"]).shape[:2]
pts = pd.read_csv(fila["cyt"], header=None).values

print(f"Recorte: {w}x{h}")
print(f"Contorno x: {pts[:,0].min():.0f} a {pts[:,0].max():.0f} | y: {pts[:,1].min():.0f} a {pts[:,1].max():.0f}")
print("Puntos en el núcleo:", len(pd.read_csv(fila["nuc"], header=None)))

El contorno cae dentro del tamaño del recorte, así que las coordenadas ya vienen relativas a
cada célula. No hay que corregir nada, las máscaras se pueden dibujar directo.

## El problema del reparto de datos

Acá está el punto central de toda esta fase.

Cuando se entrena un modelo, las fotos se reparten en dos grupos: uno para que aprenda y otro
para tomarle examen. Lo normal sería repartirlas al azar. El problema es que si una célula queda
en el examen y su hermana del mismo cúmulo quedó en el entrenamiento, el modelo ya vio algo casi
idéntico y aprueba sin haber aprendido nada. El resultado sale inflado.

Vamos a medir cuánto pasa eso, y después armamos el reparto correcto, que separa láminas
completas en vez de células sueltas.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

# 1) Reparto al azar, que es lo que se suele hacer
tr, te = train_test_split(df, test_size=0.2, stratify=df["clase"], random_state=42)
laminas_tr = set(tr["lamina"])
contaminadas = te["lamina"].isin(laminas_tr).mean()
print(f"Células de prueba con al menos una hermana en entrenamiento: {contaminadas:.1%}")

# 2) Reparto agrupado: las celulas de una misma lamina caen siempre juntas.
#    StratifiedGroupKFold ademas intenta mantener las clases balanceadas entre pliegues.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1
for k, (_, idx_val) in enumerate(sgkf.split(df, df["clase"], groups=df["lamina"])):
    df.loc[df.index[idx_val], "fold"] = k

print("\nDistribución por pliegue:")
print(df.groupby(["fold", "clase"]).size().unstack(fill_value=0))

# Control de que el agrupamiento se respeto
chequeo = df.groupby("lamina")["fold"].nunique()
print("\nLáminas repartidas entre pliegues:", (chequeo > 1).sum(), "(debe ser 0)")

**89.8%.** Con el reparto al azar, casi nueve de cada diez células del examen tienen una hermana
en el entrenamiento. Esto explica por qué en la literatura se reportan aciertos del 99% en este
dataset, y por qué esos números no significan lo que parecen.

Los cinco pliegues quedan algo desparejos (por ejemplo Disqueratóticas va de 134 a 204 según el
pliegue) porque las láminas no se pueden partir y algunas traen muchas células. No es grave,
pero por eso conviene usar métricas que promedien por clase en vez de accuracy pelada.

De acá en adelante, todos los experimentos usan estos mismos pliegues. Así las comparaciones
entre modelos son justas.

## Revisar las imágenes

Comprobamos que ninguna esté dañada y vemos de qué tamaño son.

In [ ]:
from PIL import Image

anchos, altos, rotas = [], [], []
for p in df["img"]:
    try:
        with Image.open(p) as im:
            im.verify()          # detecta archivos corruptos
        with Image.open(p) as im:
            w, h = im.size
        anchos.append(w); altos.append(h)
    except Exception:
        anchos.append(-1); altos.append(-1); rotas.append(p)

df["w"], df["h"] = anchos, altos

print("Imágenes ilegibles:", len(rotas))
print(df[["w", "h"]].describe().round(1))

ar = df["w"] / df["h"]
print("\nRelación de aspecto (ancho/alto):", round(ar.min(), 2), "a", round(ar.max(), 2))
print("Duplicados exactos de nombre:", df["stem"].duplicated().sum())
print("Recortes muy pequeños (menos de 80 px de lado):", ((df.w < 80) | (df.h < 80)).sum())

Ninguna imagen está dañada. Los recortes miden en promedio 215x191, pero van desde 62x48 hasta
531x553, casi diez veces de diferencia entre el más chico y el más grande.

Esto importa más de lo que parece. Cuando escalemos todo a un tamaño fijo para entrenar, las
células chicas se van a agrandar y las grandes se van a encoger, y ahí se pierde el tamaño real
de la célula, que es justamente uno de los rasgos que las distingue. Vale la pena guardar el
tamaño original como dato aparte para poder usarlo después.

## Revisar las anotaciones

Primero contamos qué tipos de archivo `.dat` hay, para confirmar que son exactamente dos por
célula y que no haya alguno suelto que no estemos tomando en cuenta.

In [ ]:
import collections

extras = collections.Counter()
for c in CLASES:
    cropped = os.path.join(path, c, c, "CROPPED")
    for f in os.listdir(cropped):
        if f.endswith(".dat"):
            suf = f.rsplit("_", 1)[-1]     # nos quedamos con lo de despues del ultimo guion bajo
            extras[suf] += 1

print("Terminaciones de los .dat:", dict(extras))

Ahora la comprobación visual, que es la que no se puede saltar. Dibujamos los contornos encima
de las fotos: verde el borde de la célula, rojo el núcleo. Si las coordenadas estuvieran
invertidas o corridas, todo lo que hagamos después saldría mal y sería difícil darse cuenta.

Abajo de cada foto va la máscara, que es la misma información pero en forma de regiones
pintadas. Esa máscara es lo que después le vamos a dar al modelo de segmentación.

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt

def contorno(ruta):
    return np.loadtxt(ruta, delimiter=",").astype(np.int32)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for j, clase in enumerate(df["clase"].unique()):
    fila = df[df.clase == clase].iloc[0]
    img = cv2.cvtColor(cv2.imread(fila["img"]), cv2.COLOR_BGR2RGB)
    cyt, nuc = contorno(fila["cyt"]), contorno(fila["nuc"])

    # Contornos dibujados encima de la foto
    dib = img.copy()
    cv2.polylines(dib, [cyt], True, (0, 255, 0), 2)
    cv2.polylines(dib, [nuc], True, (255, 0, 0), 2)

    # Mascara: 0 fondo, 1 citoplasma, 2 nucleo (el nucleo se pinta encima)
    mask = np.zeros(img.shape[:2], np.uint8)
    cv2.fillPoly(mask, [cyt], 1)
    cv2.fillPoly(mask, [nuc], 2)

    axes[0, j].imshow(dib); axes[0, j].set_title(clase, fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(mask); axes[1, j].axis("off")

plt.tight_layout(); plt.show()

Por último calculamos la razón entre el área del núcleo y el área de la célula. Es la medida que
más mira un especialista al microscopio: en las células anormales el núcleo tiende a ocupar más
espacio.

Sirve para dos cosas. Una, comprobar que la anotación tenga sentido (no debería haber núcleos
más grandes que su propia célula). Dos, ver si esta medida por sí sola ya separa las clases,
porque si es así nos va a servir bastante más adelante.

In [ ]:
def area(ruta):
    return cv2.contourArea(contorno(ruta).reshape(-1, 1, 2))

df["area_cyt"] = df["cyt"].apply(area)
df["area_nuc"] = df["nuc"].apply(area)
df["nc"] = df["area_nuc"] / df["area_cyt"]

# Chequeos de coherencia
print("Casos raros (núcleo mayor que la célula):", (df["nc"] > 1).sum())
print("Áreas en cero:", (df[["area_cyt", "area_nuc"]] == 0).any(axis=1).sum())

print("\nRazón núcleo/célula por clase:")
print(df.groupby("clase")["nc"].describe()[["mean", "50%", "min", "max"]].round(3))

## Guardar el índice

Este CSV es el resultado de toda la fase: el mapa completo de qué célula es cada una, de qué
lámina viene, en qué pliegue le tocó y sus medidas básicas. Todo lo que sigue se construye
encima de esto.

Lo guardamos en Drive para no tener que repetir la revisión cada vez que se cierre la sesión
de Colab.

In [ ]:
DEST = "/content/drive/MyDrive/sipakmed"
os.makedirs(DEST, exist_ok=True)

df.to_csv(f"{DEST}/indice_celulas.csv", index=False)
print("Guardado. Filas:", len(df), "| Columnas:", list(df.columns))

## Resumen de la fase 1

Lo que encontramos:

- 4049 células, cinco clases bastante balanceadas, ninguna imagen dañada y ningún contorno faltante.
- Las células salen de solo 966 láminas, con 4.2 en promedio por lámina.
- Repartiendo al azar, el 89.8% de las células de prueba tienen una hermana en el entrenamiento.
  Este es el hallazgo principal y es el que justifica todo el enfoque del trabajo.
- Los contornos vienen en coordenadas del recorte, así que las máscaras se pueden usar directo.
- Los tamaños de los recortes varían mucho, y esa información se pierde al escalar.

Queda listo el reparto en cinco pliegues por lámina, que es lo que vamos a usar en todos los
experimentos.

**Lo que sigue (fase 2):** entrenar el mismo modelo dos veces, una con el reparto al azar y otra
con el reparto por lámina, y medir cuánto baja el acierto. Esa diferencia es el resultado central.